In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DATASET_PATH = "/content/drive/MyDrive/IITG dataset/cleaned_dataset"
DATA_PATH = os.path.join(DATASET_PATH, "data")
METADATA_PATH = os.path.join(DATASET_PATH, "metadata.csv")

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Data folder exists:", os.path.exists(DATA_PATH))
print("Metadata exists:", os.path.exists(METADATA_PATH))

Dataset exists: True
Data folder exists: True
Metadata exists: True


In [3]:
import pandas as pd

csv_files = [
    f for f in os.listdir(DATA_PATH)
    if f.lower().endswith(".csv")
]

metadata = pd.read_csv(METADATA_PATH)

print("Number of battery files:", len(csv_files))
print("Metadata shape:", metadata.shape)

Number of battery files: 7600
Metadata shape: (7565, 10)


In [4]:
# Day 8 - Inspect battery data structure

sample_file = os.path.join(DATA_PATH, csv_files[0])
sample_df = pd.read_csv(sample_file)

print("Sample file:", csv_files[0])
print("Shape:", sample_df.shape)

print("\nColumns:")
for col in sample_df.columns:
    print("-", col)

print("\nFirst 5 rows:")
display(sample_df.head())

Sample file: 06664.csv
Shape: (1403, 6)

Columns:
- Voltage_measured
- Current_measured
- Temperature_measured
- Current_charge
- Voltage_charge
- Time

First 5 rows:


,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time
0,3.515781,-0.000839,33.942370,0.000,-0.007,0.000
1,3.194762,-3.612513,33.866384,-3.629,1.461,2.453
2,3.578757,1.515107,33.857983,1.507,4.289,9.422
3,3.635699,1.515922,33.863405,1.507,4.359,16.187
4,3.665379,1.515677,33.799425,1.507,4.392,22.937


In [5]:
# DAY 8 - Task 1: Missing and Invalid Reading Audit

import numpy as np

# Convert numeric columns temporarily for checking
numeric_cols = [
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured",
    "Current_charge",
    "Voltage_charge",
    "Time"
]

audit = []

for col in numeric_cols:
    values = pd.to_numeric(sample_df[col], errors="coerce")

    audit.append({
        "Parameter": col,
        "Missing": values.isna().sum(),
        "Infinite": np.isinf(values).sum(),
        "Minimum": values.min(),
        "Maximum": values.max()
    })

audit_df = pd.DataFrame(audit)

display(audit_df)

,Parameter,Missing,Infinite,Minimum,Maximum
0,Voltage_measured,0,0,3.194762,4.203718
1,Current_measured,0,0,-3.612513,1.521529
2,Temperature_measured,0,0,23.720182,33.942370
3,Current_charge,0,0,-3.629000,1.507000
4,Voltage_charge,0,0,-0.007000,4.926000
5,Time,0,0,0.000000,9930.265000


In [6]:
print("Metadata columns:")
print(metadata.columns.tolist())

Metadata columns:
['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']


In [7]:
# DAY 8 - Metadata / Battery / Test consistency

print("Number of metadata rows:", len(metadata))

print("\nUnique batteries:", metadata["battery_id"].nunique())
print("Unique tests:", metadata["test_id"].nunique())
print("Unique files:", metadata["filename"].nunique())

print("\nBattery type distribution:")
print(metadata["type"].value_counts(dropna=False))

print("\nSamples per battery:")
display(metadata["battery_id"].value_counts().describe())

Number of metadata rows: 7565

Unique batteries: 34
Unique tests: 616
Unique files: 7565

Battery type distribution:
type
charge       2815
discharge    2794
impedance    1956
Name: count, dtype: int64

Samples per battery:


,count
count,34.000000
mean,222.500000
std,172.979461
min,62.000000
25%,97.000000
50%,173.500000
75%,275.000000
max,616.000000


In [8]:
# DAY 8 - Target / Type Consistency

print("Missing values in metadata:")
display(metadata.isna().sum())

print("\nBattery type distribution:")
display(metadata["type"].value_counts(dropna=False))

print("\nType distribution by battery:")
type_consistency = pd.crosstab(metadata["battery_id"], metadata["type"])

display(type_consistency.head(10))

Missing values in metadata:


,0
type,0
start_time,0
ambient_temperature,0
battery_id,0
test_id,0
uid,0
filename,0
Capacity,4771
Re,5609
Rct,5609



Battery type distribution:


,count
type,
charge,2815
discharge,2794
impedance,1956



Type distribution by battery:


type,charge,discharge,impedance
battery_id,,,
B0005,170,168,278
B0006,170,168,278
B0007,170,168,278
B0018,134,132,53
B0025,31,28,21
B0026,31,28,21
B0027,31,28,21
B0028,31,28,21
B0029,40,40,17


In [9]:
# DAY 8 - Sequence / Test Length Assessment

test_lengths = (
    metadata.groupby(["battery_id", "test_id"])
    .size()
    .reset_index(name="records")
)

print("Number of battery-test sequences:", len(test_lengths))

print("\nSequence length statistics:")
display(test_lengths["records"].describe())

print("\nShortest sequences:")
display(test_lengths.nsmallest(10, "records"))

print("\nLongest sequences:")
display(test_lengths.nlargest(10, "records"))

Number of battery-test sequences: 7565

Sequence length statistics:


,records
count,7565.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0



Shortest sequences:


,battery_id,test_id,records
0,B0005,0,1
1,B0005,1,1
2,B0005,2,1
3,B0005,3,1
4,B0005,4,1
5,B0005,5,1
6,B0005,6,1
7,B0005,7,1
8,B0005,8,1
9,B0005,9,1



Longest sequences:


,battery_id,test_id,records
0,B0005,0,1
1,B0005,1,1
2,B0005,2,1
3,B0005,3,1
4,B0005,4,1
5,B0005,5,1
6,B0005,6,1
7,B0005,7,1
8,B0005,8,1
9,B0005,9,1


In [10]:
# DAY 8 - Sequence / Cycle Consistency

# Number of recorded tests/files for each battery and test type
sequence_summary = (
    metadata.groupby(["battery_id", "type"])
    .size()
    .reset_index(name="number_of_sequences")
)

print("Sequences/tests per battery and type:")
display(sequence_summary.head(20))

print("\nSequence count statistics:")
display(sequence_summary["number_of_sequences"].describe())

Sequences/tests per battery and type:


,battery_id,type,number_of_sequences
0,B0005,charge,170
1,B0005,discharge,168
2,B0005,impedance,278
3,B0006,charge,170
4,B0006,discharge,168
5,B0006,impedance,278
6,B0007,charge,170
7,B0007,discharge,168
8,B0007,impedance,278
9,B0018,charge,134



Sequence count statistics:


,number_of_sequences
count,102.000000
mean,74.166667
std,62.929960
min,12.000000
25%,28.000000
50%,47.500000
75%,102.000000
max,278.000000


In [11]:
# Check test ID continuity within each battery

gap_results = []

for battery, group in metadata.groupby("battery_id"):
    test_ids = sorted(pd.to_numeric(group["test_id"], errors="coerce").dropna().unique())

    if len(test_ids) > 1:
        expected = set(range(int(min(test_ids)), int(max(test_ids)) + 1))
        missing = sorted(expected - set(test_ids))

        gap_results.append({
            "battery_id": battery,
            "first_test": min(test_ids),
            "last_test": max(test_ids),
            "number_of_tests": len(test_ids),
            "missing_test_ids": len(missing)
        })

gap_df = pd.DataFrame(gap_results)

display(gap_df)

,battery_id,first_test,last_test,number_of_tests,missing_test_ids
0,B0005,0,615,616,0
1,B0006,0,615,616,0
2,B0007,0,615,616,0
3,B0018,0,318,319,0
4,B0025,0,79,80,0
5,B0026,0,79,80,0
6,B0027,0,79,80,0
7,B0028,0,79,80,0
8,B0029,0,96,97,0
9,B0030,0,96,97,0


In [12]:
# DAY 8 - Future Information Leakage Check

print("Potential time/order columns:")
for col in metadata.columns:
    if any(word in col.lower() for word in ["time", "start", "date", "cycle", "test"]):
        print("-", col)

print("\nColumns that may represent future/target information:")
for col in metadata.columns:
    if any(word in col.lower() for word in ["capacity", "re", "rct", "target", "label"]):
        print("-", col)

Potential time/order columns:
- start_time
- test_id

Columns that may represent future/target information:
- ambient_temperature
- Capacity
- Re
- Rct


DAY 8 - CLEANING AND ALIGNMENT PLAN

1. Missing values:
   Identify missing measurements and quantify them before deciding whether
   to remove, interpolate, or retain the affected records.

2. Invalid readings:
   Flag physically implausible or non-finite voltage, current,
   temperature and time values for further inspection.

3. Duplicate records:
   Identify duplicate measurements and verify whether they are true
   duplicates before removal.

4. Sequence/cycle alignment:
   Preserve the chronological order of measurements using the available
   time/test information. Do not mix measurements from different batteries
   or tests.

5. Partial sequences:
   Flag unusually short or incomplete sequences for inspection rather
   than automatically deleting them.

6. Target consistency:
   Keep target variables separate from input features and verify that
   target values correspond to the correct battery/test.

7. Leakage prevention:
   Use only information available at the prediction time. Future-cycle
   measurements and future target information must not be used as input
   features.